这是数据预处理的第三部分:
adata是完成基因插补的Xenium合成的Visium HD数据, bdata是同类可参考的Visium HD数据;cdata是我们这部分处理得到的结果；
先进行数据检查，对齐共有基因，检查adata和bdata的UMI分布律在细胞区域和背景区域差异情况，这将作为后续处理的参数设置的参考；
进行降采样处理：使用高斯模糊来模拟透化，使用gamma-poisson分布来近似负二项分布，采用向量化的数组计算来提高计算效率，
得到的csr保存为cdata.X，最后进行ROI选定和结果检查。

In [ ]:
import scanpy as sc
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import scipy.sparse as sp
from scipy.ndimage import distance_transform_edt
import torchvision.transforms.functional as TF

# 优化 CUDA 显存分配器，减少碎片化
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# adata: 完成基因集插补的Xenium合成的Visium HD数据; bdata:用于参考表达谱的Visium HD数据
adata = sc.read_h5ad('/root/autodl-tmp/PRAD_sys_2um.h5ad')
bdata = sc.read_h5ad('/root/autodl-tmp/visium_new.h5ad')

In [ ]:
# 检查 adata 数据完整性

# 1. 检查表达矩阵
print(f"adata.X shape: {adata.X.shape}")
print(f"adata.X total counts: {adata.X.sum():.2f}")

# 2. 检查 cell_id 列
unique_cells = adata.obs['cell_id'].unique()
print(f"Total unique cell_ids: {len(unique_cells)}")
print(f"Sample cell_ids: {unique_cells[:10]}")
print(f"Missing/NaN cell_ids count: {adata.obs['cell_id'].isna().sum()}")

# 3. 检查提取逻辑的中间结果
raw_bin_depths = np.array(adata.X.sum(axis=1)).flatten()
print(f"Bins with depth > 0: {(raw_bin_depths > 0).sum()} / {len(raw_bin_depths)}")

# 4. 检查最终输出给绘图的数组
print(f"Final depths_a length: {len(depths_a)}")
if len(depths_a) > 0:
    print(f"depths_a Min: {depths_a.min():.2f}, Max: {depths_a.max():.2f}")
else:
    print("WARNING: depths_a is empty! Plotting will fail/skip.")

In [ ]:
#  数据检查和处理: 检查adata和bdata的共有基因数据, 细胞区域和非细胞区域的表达差异度
# ==========================================
# Step 1: Align Gene Spaces
# ==========================================
common_genes = adata.var_names.intersection(bdata.var_names)
print(f"Number of common genes: {len(common_genes)}")

X_xe = adata[:, common_genes].X
X_hd = bdata[:, common_genes].X

# ==========================================
# Step 2: Foreground vs Background Masks (Fixed)
# ==========================================
labels = bdata.obs['labels_he_expanded'].astype(str)
# Convert Pandas Series to NumPy array for SciPy sparse indexing
fg_mask_hd = ((labels != '0') & (labels != '') & (labels != 'nan')).to_numpy()
bg_mask_hd = ~fg_mask_hd

n_fg_hd = fg_mask_hd.sum()
n_bg_hd = bg_mask_hd.sum()
n_xe = X_xe.shape[0]

# ==========================================
# Step 3: Compute Statistics (Sparse Optimized)
# ==========================================
fg_hd_sums = np.asarray(X_hd[fg_mask_hd].sum(axis=0)).flatten()
fg_hd_means = fg_hd_sums / n_fg_hd

bg_hd_sums = np.asarray(X_hd[bg_mask_hd].sum(axis=0)).flatten()
bg_hd_means = bg_hd_sums / n_bg_hd

xe_sums = np.asarray(X_xe.sum(axis=0)).flatten()
xe_means = xe_sums / n_xe

# ==========================================
# Compile and Display Results
# ==========================================
stats_df = pd.DataFrame({
    'gene_id': common_genes,
    'hd_fg_mean': fg_hd_means,
    'hd_bg_mean': bg_hd_means,
    'xe_mean': xe_means
})

print(f"\nReal HD - Foreground Bins: {n_fg_hd}, Background Bins: {n_bg_hd}")
print(f"Xenium - Total Bins: {n_xe}")

print("\nGlobal Depth (Common Genes Only):")
print(f"Real HD Foreground Depth: {fg_hd_means.sum():.2f}")
print(f"Real HD Background Depth: {bg_hd_means.sum():.2f}")
print(f"Xenium Depth: {xe_means.sum():.2f}")

print("\nGene-level Statistics Summary:")
print(stats_df.describe())



In [ ]:
# 使用高斯模糊和Gamma-Poisson分布式处理adata的表达谱,作为后续训练模型的输入部分[H,W,C]
# ==========================================
# Step 1: Sparse-Native Parameter Extraction
# ==========================================
common_genes = adata.var_names.intersection(bdata.var_names)
X_src = adata[:, common_genes].X.tocsr()  # 确保是 CSR 格式加速行切片
X_tgt = bdata[:, common_genes].X
C = len(common_genes)

labels = bdata.obs['labels_he_expanded'].astype(str).values
fg_mask = (labels != '0') & (labels != '') & (labels != 'nan')
bg_mask = ~fg_mask

fg_vec = sp.csr_matrix(fg_mask.astype(np.float32)).T
bg_vec = sp.csr_matrix(bg_mask.astype(np.float32)).T

tgt_fg_mean = (X_tgt.T @ fg_vec).toarray().flatten() / max(fg_mask.sum(), 1)
tgt_bg_mean = (X_tgt.T @ bg_vec).toarray().flatten() / max(bg_mask.sum(), 1)
src_mean = np.array(X_src.sum(axis=0)).flatten() / X_src.shape[0]

epsilon = 1e-8
scale_factors = torch.tensor(tgt_fg_mean / (src_mean + epsilon), dtype=torch.float32).view(C, 1, 1)
bg_base = torch.tensor(tgt_bg_mean, dtype=torch.float32).view(C, 1, 1)
theta = torch.full((C, 1, 1), 10.0, dtype=torch.float32)

# ==========================================
# Step 2: Spatial Mapping & 2D Distance Field
# ==========================================
coords = adata.obsm['spatial']
x_coords = np.round(coords[:, 0] - coords[:, 0].min()).astype(int)
y_coords = np.round(coords[:, 1] - coords[:, 1].min()).astype(int)

W, H = x_coords.max() + 1, y_coords.max() + 1
tile_size = 1024

tile_x_idx = x_coords // tile_size
tile_y_idx = y_coords // tile_size

spot_totals = np.array(X_src.sum(axis=1)).flatten()
tissue_spots = spot_totals > 0
global_mask = np.zeros((H, W), dtype=bool)
global_mask[y_coords[tissue_spots], x_coords[tissue_spots]] = True

alpha = 0.05 
distances = distance_transform_edt(~global_mask)
decay_map = torch.tensor(np.exp(-alpha * distances), dtype=torch.float32)

# ==========================================
# Step 3: Tile Processing & GPU Sparsification
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
scale_factors = scale_factors.to(device)
bg_base = bg_base.to(device)
theta = theta.to(device)
decay_map = decay_map.to(device)

out_rows, out_cols, out_data = [], [], []
chunk_size = 500  # 进一步减小 chunk_size 以适应当前剩余显存

for ty in range(tile_y_idx.max() + 1):
    for tx in range(tile_x_idx.max() + 1):
        
        in_tile = (tile_x_idx == tx) & (tile_y_idx == ty)
        has_spots = in_tile.any()
        
        if has_spots:
            local_x = x_coords[in_tile] - (tx * tile_size)
            local_y = y_coords[in_tile] - (ty * tile_size)
            
        y_start, y_end = ty * tile_size, min((ty + 1) * tile_size, H)
        x_start, x_end = tx * tile_size, min((tx + 1) * tile_size, W)
        
        local_decay = torch.zeros((tile_size, tile_size), dtype=torch.float32, device=device)
        local_decay[:y_end-y_start, :x_end-x_start] = decay_map[y_start:y_end, x_start:x_end]
        
        # 将张量初始化移入 Chunk 循环，严格控制峰值显存
        for c_start in range(0, C, chunk_size):
            c_end = min(c_start + chunk_size, C)
            current_chunk_size = c_end - c_start
            
            tile_chunk = torch.zeros((current_chunk_size, tile_size, tile_size), dtype=torch.float32, device=device)
            
            if has_spots:
                # 仅提取当前基因块的数据并送入 GPU
                chunk_data = X_src[in_tile, c_start:c_end].toarray()
                tile_chunk[:, local_y, local_x] = torch.tensor(chunk_data, dtype=torch.float32, device=device).T
            
            sf_chunk = scale_factors[c_start:c_end]
            bg_chunk = bg_base[c_start:c_end]
            theta_chunk = theta[c_start:c_end]
            
            local_bg = bg_chunk * local_decay.unsqueeze(0)
            
            blurred = TF.gaussian_blur(tile_chunk, kernel_size=5, sigma=[1.5, 1.5])
            expected = (blurred * sf_chunk) + local_bg
            rate = theta_chunk / (expected + epsilon)
            
            gamma_samples = torch.distributions.Gamma(theta_chunk, rate).sample()
            degraded_tile = torch.distributions.Poisson(gamma_samples).sample()
            
            c_idx, y_idx, x_idx = torch.nonzero(degraded_tile, as_tuple=True)
            vals = degraded_tile[c_idx, y_idx, x_idx]
            
            c_idx = c_idx.cpu().numpy() + c_start
            y_idx = y_idx.cpu().numpy()
            x_idx = x_idx.cpu().numpy()
            vals = vals.cpu().numpy()
            
            global_y = y_idx + (ty * tile_size)
            global_x = x_idx + (tx * tile_size)
            
            valid_mask = (global_x < W) & (global_y < H)
            global_x = global_x[valid_mask]
            global_y = global_y[valid_mask]
            c_idx = c_idx[valid_mask]
            vals = vals[valid_mask]
            
            global_pixel_idx = global_y * W + global_x
            
            out_rows.append(global_pixel_idx)
            out_cols.append(c_idx)
            out_data.append(vals)
            
            del tile_chunk, sf_chunk, bg_chunk, theta_chunk, local_bg, blurred, expected, rate, gamma_samples, degraded_tile
            
        del local_decay

# ==========================================
# Step 4: Final Assembly
# ==========================================
final_rows = np.concatenate(out_rows)
final_cols = np.concatenate(out_cols)
final_data = np.concatenate(out_data)

degraded_sparse_matrix = sp.coo_matrix(
    (final_data, (final_rows, final_cols)), 
    shape=(H * W, C)
).tocsr()

In [ ]:
# 最后拷贝adata为cdata, 使用csr替代cdata.X, cdata即为后续模型的输入 
# ==========================================
# Create cdata (Degraded AnnData)
# ==========================================
original_pixel_indices = y_coords * W + x_coords

cdata = adata[:, common_genes].copy()
cdata.X = degraded_sparse_matrix[original_pixel_indices, :]
cdata.uns['is_degraded'] = True

In [ ]:
# 注意: 和上一个kernel中的效果一样,是使用函数封装后的处理函数(一式两份,便于调试)

def degrade_spatial_data(adata, bdata, tile_size=512, kernel_size=5, sigma=1.5, theta_val=10.0, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # ==========================================
    # Step 1: Parameter Extraction
    # ==========================================
    common_genes = adata.var_names.intersection(bdata.var_names)
    X_src = adata[:, common_genes].X
    X_tgt = bdata[:, common_genes].X
    C = len(common_genes)

    labels = bdata.obs['labels_he_expanded'].astype(str).values
    fg_mask = (labels != '0') & (labels != '') & (labels != 'nan')
    bg_mask = ~fg_mask

    fg_vec = sp.csr_matrix(fg_mask.astype(np.float32)).T
    bg_vec = sp.csr_matrix(bg_mask.astype(np.float32)).T

    tgt_fg_mean = (X_tgt.T @ fg_vec).toarray().flatten() / max(fg_mask.sum(), 1)
    tgt_bg_mean = (X_tgt.T @ bg_vec).toarray().flatten() / max(bg_mask.sum(), 1)
    src_mean = np.array(X_src.sum(axis=0)).flatten() / X_src.shape[0]

    epsilon = 1e-8
    scale_factors = torch.tensor(tgt_fg_mean / (src_mean + epsilon), dtype=torch.float32).view(C, 1, 1).to(device)
    background = torch.tensor(tgt_bg_mean, dtype=torch.float32).view(C, 1, 1).to(device)
    theta = torch.full((C, 1, 1), theta_val, dtype=torch.float32).to(device)

    # ==========================================
    # Step 2: Spatial Coordinate Mapping
    # ==========================================
    coords = adata.obsm['spatial']
    x_coords = np.round(coords[:, 0] - coords[:, 0].min()).astype(int)
    y_coords = np.round(coords[:, 1] - coords[:, 1].min()).astype(int)

    W, H = x_coords.max() + 1, y_coords.max() + 1
    tile_x_idx = x_coords // tile_size
    tile_y_idx = y_coords // tile_size

    # ==========================================
    # Step 3: Tile Processing & GPU Sparsification
    # ==========================================
    out_rows, out_cols, out_data = [], [], []

    for ty in range(tile_y_idx.max() + 1):
        for tx in range(tile_x_idx.max() + 1):
            tile_tensor = torch.zeros((C, tile_size, tile_size), dtype=torch.float32)
            in_tile = (tile_x_idx == tx) & (tile_y_idx == ty)
            
            if in_tile.any():
                local_x = x_coords[in_tile] - (tx * tile_size)
                local_y = y_coords[in_tile] - (ty * tile_size)
                tile_tensor[:, local_y, local_x] = torch.tensor(X_src[in_tile].toarray(), dtype=torch.float32).T
                
            tile_tensor = tile_tensor.to(device)
            
            blurred = TF.gaussian_blur(tile_tensor, kernel_size=kernel_size, sigma=[sigma, sigma])
            expected = (blurred * scale_factors) + background
            rate = theta / (expected + epsilon)
            
            gamma_samples = torch.distributions.Gamma(theta, rate).sample()
            degraded_tile = torch.distributions.Poisson(gamma_samples).sample()
            
            c_idx, y_idx, x_idx = torch.nonzero(degraded_tile, as_tuple=True)
            vals = degraded_tile[c_idx, y_idx, x_idx]
            
            global_y = y_idx.cpu().numpy() + (ty * tile_size)
            global_x = x_idx.cpu().numpy() + (tx * tile_size)
            c_idx = c_idx.cpu().numpy()
            vals = vals.cpu().numpy()
            
            valid_mask = (global_x < W) & (global_y < H)
            
            out_rows.append((global_y[valid_mask] * W) + global_x[valid_mask])
            out_cols.append(c_idx[valid_mask])
            out_data.append(vals[valid_mask])

    # ==========================================
    # Step 4: Final Assembly & cdata Construction
    # ==========================================
    degraded_sparse_matrix = sp.coo_matrix(
        (np.concatenate(out_data), (np.concatenate(out_rows), np.concatenate(out_cols))), 
        shape=(H * W, C)
    ).tocsr()

    original_pixel_indices = y_coords * W + x_coords
    
    cdata = adata[:, common_genes].copy()
    cdata.X = degraded_sparse_matrix[original_pixel_indices, :]
    cdata.uns['degradation_params'] = {'kernel_size': kernel_size, 'sigma': sigma, 'theta': theta_val}

    return cdata

# 使用示例：
# cdata = degrade_spatial_data(adata, bdata, tile_size=512, kernel_size=5, sigma=1.5, theta_val=10.0)

In [ ]:
# 储存cdata为h5ad格式
cdata.write_h5ad('degraded_sys_visium.h5ad')

In [ ]:
# 检查adata和cdata之间的表达谱差异度
common_genes = adata.var_names.intersection(bdata.var_names)
# ==========================================
# Step 1: Global Statistics Comparison
# ==========================================
adata_X = adata[:, common_genes].X
cdata_X = cdata.X

def print_stats(name, matrix):
    sparsity = 1.0 - (matrix.nnz / (matrix.shape[0] * matrix.shape[1]))
    total_counts = matrix.sum()
    mean_counts_per_spot = matrix.sum(axis=1).mean()
    print(f"--- {name} ---")
    print(f"Sparsity: {sparsity:.4f}")
    print(f"Total Counts: {total_counts:.0f}")
    print(f"Mean Counts/Spot: {mean_counts_per_spot:.2f}\n")

print_stats("Original (adata)", adata_X)
print_stats("Degraded (cdata)", cdata_X)

In [ ]:
#绘图比较adata,bdata,cdata的UMI分布律

# ==========================================
# Step 1 & 2: Target Data Extraction & Stats
# ==========================================
bdata_X = bdata[:, common_genes].X
print_stats("Target (bdata)", bdata_X)

# ==========================================
# Step 3: Spot-level Distribution Comparison
# ==========================================
counts_adata = np.array(adata_X.sum(axis=1)).flatten()
counts_cdata = np.array(cdata_X.sum(axis=1)).flatten()
counts_bdata = np.array(bdata_X.sum(axis=1)).flatten()

# 截断长尾以优化可视化效果 (可根据实际数据调整 range_max)
range_max = np.percentile(counts_bdata, 98) 

plt.figure(figsize=(10, 5), dpi=200)

plt.hist(counts_adata, bins=60, alpha=0.5, label='Original (adata)', density=True, range=(0, range_max))
plt.hist(counts_bdata, bins=60, alpha=0.5, label='Target (bdata)', density=True, range=(0, range_max), color='red')
plt.hist(counts_cdata, bins=60, alpha=0.5, label='Simulated (cdata)', density=True, range=(0, range_max), color='green')

plt.legend()
plt.xlabel('Total Counts per Spot')
plt.ylabel('Density')
plt.title('Distribution of Total Counts per Spot (Common Genes)')
plt.show()

In [ ]:
# 检查各数据的实际坐标边界
print("bdata X range:", bdata.obsm['spatial'][:, 0].min(), "-", bdata.obsm['spatial'][:, 0].max())
print("bdata Y range:", bdata.obsm['spatial'][:, 1].min(), "-", bdata.obsm['spatial'][:, 1].max())
print("cdata X range:", cdata.obsm['spatial'][:, 0].min(), "-", cdata.obsm['spatial'][:, 0].max())
print("cdata Y range:", cdata.obsm['spatial'][:, 1].min(), "-", cdata.obsm['spatial'][:, 1].max())

In [ ]:
# 绘图检查:细胞核,细胞,表达谱强度的分布情况
# ==========================================
# Step 1: ROI Definition & Data Extraction
# ==========================================
x_min, x_max = 800, 1000
y_min, y_max = 800, 1000

bin_x = (adata.obsm['spatial'][:, 0] / 2).astype(int)
bin_y = (adata.obsm['spatial'][:, 1] / 2).astype(int)
in_roi = (bin_x >= x_min) & (bin_x < x_max) & (bin_y >= y_min) & (bin_y < y_max)

rx = bin_x[in_roi]
ry = bin_y[in_roi]

# 提取形态学标注 (cdata 是 adata 的 copy，obs 相同)
r_cell = adata.obs['cell_id'].values[in_roi]
r_nuc = adata.obs['synthesis_nucleus_id'].values[in_roi]

# ==========================================
# Step 2: Absolute Expression Aggregation
# ==========================================
adata_roi_counts = np.array(adata[in_roi].X.sum(axis=1)).flatten()
cdata_roi_counts = np.array(cdata[in_roi].X.sum(axis=1)).flatten()

img_shape = (y_max - y_min, x_max - x_min)
img_adata = np.zeros(img_shape, dtype=np.float32)
img_cdata = np.zeros(img_shape, dtype=np.float32)

img_adata[ry - y_min, rx - x_min] = adata_roi_counts
img_cdata[ry - y_min, rx - x_min] = cdata_roi_counts

# ==========================================
# Step 3: Consistent Color Scaling
# ==========================================
shared_vmax = max(img_adata.max(), img_cdata.max())

# ==========================================
# Step 4: Spatial Visualization
# ==========================================
is_bg = r_cell == "0"
is_nuc = r_nuc != "0"
is_cell = (r_cell != "0") & (r_nuc == "0")

fig, axes = plt.subplots(1, 2, figsize=(16, 8), dpi=200)
extent_args = [x_min, x_max, y_min, y_max]

# 绘制原始数据
im0 = axes[0].imshow(img_adata, cmap='gray', origin='lower', extent=extent_args, vmin=0, vmax=shared_vmax)
axes[0].scatter(rx[is_bg] + 0.5, ry[is_bg] + 0.5, c='gray', s=0.5, alpha=0.3)
axes[0].scatter(rx[is_cell] + 0.5, ry[is_cell] + 0.5, c='green', s=1, alpha=0.9)
axes[0].scatter(rx[is_nuc] + 0.5, ry[is_nuc] + 0.5, c='yellow', s=1, alpha=0.9)
axes[0].set_title("Original Absolute Expression (adata)")

# 绘制退化数据
im1 = axes[1].imshow(img_cdata, cmap='gray', origin='lower', extent=extent_args, vmin=0, vmax=shared_vmax)
axes[1].scatter(rx[is_bg] + 0.5, ry[is_bg] + 0.5, c='gray', s=0.5, alpha=0.3)
axes[1].scatter(rx[is_cell] + 0.5, ry[is_cell] + 0.5, c='green', s=1, alpha=0.9)
axes[1].scatter(rx[is_nuc] + 0.5, ry[is_nuc] + 0.5, c='yellow', s=1, alpha=0.9)
axes[1].set_title("Degraded Absolute Expression (cdata)")

# 共享 Colorbar
cbar = fig.colorbar(im1, ax=axes.ravel().tolist(), shrink=0.7)
cbar.set_label('Total Counts per Bin (Absolute)')

plt.show()

In [ ]:
# 仅保留表达谱counts的灰度图,直观比较处理前后合成数据的变化情况

# ==========================================
# Step 1: ROI Definition & Coordinate Mapping
# ==========================================
x_min, x_max = 800, 1000
y_min, y_max = 800, 1000

bin_x = (adata.obsm['spatial'][:, 0] / 2).astype(int)
bin_y = (adata.obsm['spatial'][:, 1] / 2).astype(int)
in_roi = (bin_x >= x_min) & (bin_x < x_max) & (bin_y >= y_min) & (bin_y < y_max)

rx = bin_x[in_roi]
ry = bin_y[in_roi]

# ==========================================
# Step 2 & 3: Absolute Expression to 2D Grid
# ==========================================
adata_roi_counts = np.array(adata[in_roi].X.sum(axis=1)).flatten()
cdata_roi_counts = np.array(cdata[in_roi].X.sum(axis=1)).flatten()

img_shape = (y_max - y_min, x_max - x_min)
img_adata = np.zeros(img_shape, dtype=np.float32)
img_cdata = np.zeros(img_shape, dtype=np.float32)

img_adata[ry - y_min, rx - x_min] = adata_roi_counts
img_cdata[ry - y_min, rx - x_min] = cdata_roi_counts

# ==========================================
# Step 4: 独立色阶可视化 (Decoupled Color Scales)
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(14, 7), dpi=200)
extent_args = [x_min, x_max, y_min, y_max]

# adata 使用自身的 max
im0 = axes[0].imshow(img_adata, cmap='gray', origin='lower', extent=extent_args, vmin=0, vmax=img_adata.max())
axes[0].set_title("Original Absolute Expression (adata)")
fig.colorbar(im0, ax=axes[0], shrink=0.7, label='Counts (adata scale)')

# cdata 使用自身的 max
im1 = axes[1].imshow(img_cdata, cmap='gray', origin='lower', extent=extent_args, vmin=0, vmax=img_cdata.max())
axes[1].set_title("Degraded Absolute Expression (cdata)")
fig.colorbar(im1, ax=axes[1], shrink=0.7, label='Counts (cdata scale)')

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# 可视化检查bdata和cdata(adata)的UMI分布律
# ==========================================
# Step 1 & 2: 修复分组逻辑并过滤零值
# ==========================================
def get_valid_cell_depths(adata_obj, cell_label_col):
    bin_depths = np.array(adata_obj.X.sum(axis=1)).flatten()
    df = pd.DataFrame({'cell_id': adata_obj.obs[cell_label_col].values, 'depth': bin_depths})
    
    # 过滤背景
    df = df[~df['cell_id'].isin([0, '0', None, ''])]
    
    # 添加 observed=True 避免生成大量未观察到的 0 值类别
    cell_depths = df.groupby('cell_id', observed=True)['depth'].sum().values
    
    # 必须剔除 0 值，否则 log_scale=True 会导致 KDE 计算出 NaN
    return cell_depths[cell_depths > 0]

depths_a = get_valid_cell_depths(adata, 'cell_id')
depths_b = get_valid_cell_depths(bdata, 'labels_he_expanded')
depths_c = get_valid_cell_depths(cdata, 'cell_id')

# ==========================================
# 绘制无填充的 KDE 图以展示完美重叠
# ==========================================
plt.figure(figsize=(10, 6))

# adata: 粗实线，底层
sns.kdeplot(depths_a, log_scale=True, label='adata (Scaled Xenium)', fill=False, linewidth=4, color='blue')

# bdata: 细虚线，顶层 (如果完美重叠，虚线会嵌在粗实线中间)
sns.kdeplot(depths_b, log_scale=True, label='bdata (Real Visium HD)', fill=False, linewidth=2, color='orange', linestyle='--')

# cdata: 保持半透明填充，作为对比基准
sns.kdeplot(depths_c, log_scale=True, label='cdata (Synthetic Input)', fill=True, alpha=0.3, color='green')

plt.title('Cell-level Sequencing Depth Distribution (Unfilled to reveal overlap)')
plt.xlabel('Total Counts per Cell (Log Scale)')
plt.ylabel('Density')
plt.legend()
plt.tight_layout()
plt.show()